[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/03_random_variables_and_distribution_functions/exercises.ipynb)

# Exercises — Module 03: Random Variables and Distribution Functions

20 fully solved problems in four tiers: L0 — Concept Checks (4), L1 — Foundations (6), L2 — Applications in AI/ML and Physics (6), L3 — Challenge Proofs (4). Every numeric answer is recomputed by the code cell that follows it.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

from scipy import integrate, stats

## L0 — Concept Checks

### Problem L0.1 — Can a Density Exceed 1?

**Statement**

The distribution $\text{Unif}(0, 0.1)$ has density $f(x) = 10$ on its support. Explain why this does not violate any probability axiom.

**Intuition**

A density is mass per unit length, not mass; squeezing all the mass into a short interval must push the rate up.

**Solution**

**Step 1.** The axioms constrain integrals of $f$, not values of $f$:

$$
\int_0^{0.1} 10\,dx = 1 .
$$

**Step 2.** Every event probability is such an integral, and the support is only $0.1$ long:

$$
P(a \le X \le b) = \int_a^b 10\,dx = 10(b - a) \le 1 \quad \text{for } [a,b] \subseteq [0, 0.1].
$$

**Step 3.** So no probability exceeds $1$ even though $f = 10$ everywhere on the support. Densities behave like mass densities in physics: a dense object can have density above $1$ kg/L without containing more mass than it contains.

$$
\boxed{f(x) \gt 1 \text{ is legal: only } \textstyle\int_A f \text{ must lie in } [0,1]}
$$

**Key takeaway** — PDF values are rates, not probabilities; compressing mass into a short interval necessarily pushes the density above $1$.

In [2]:
f = lambda x: 10.0
total, _ = integrate.quad(f, 0.0, 0.1)
widest = 10 * (0.1 - 0.0)
print(f"total mass       = {total:.12f}")
print(f"largest interval probability = {widest:.12f}")
assert abs(total - 1.0) < 1e-12 and widest <= 1.0

total mass       = 1.000000000000
largest interval probability = 1.000000000000


### Problem L0.2 — Which Functions Are CDFs?

**Statement**

For each function, decide whether it is a valid CDF: (a) $F(x) = \frac{1}{1 + e^{-x}}$; (b) $G(x) = \sin x$ on $[0, \pi]$ (extended by $0$ and $1$); (c) $H(x) = \mathbf{1}_{[0,\infty)}(x)\left(1 - e^{-x}\right) + \frac{1}{2}\mathbf{1}_{[1,\infty)}(x)$, suitably normalized.

**Intuition**

Run the three conditions of Theorem 4.1 down the list: monotone, limits $0$ and $1$, right-continuous.

**Solution**

**Step 1 — (a).** The logistic function is strictly increasing, continuous, with $F(-\infty) = 0$ and $F(\infty) = 1$. **Valid CDF** (the logistic distribution).

**Step 2 — (b).** On $[0, \pi]$, $\sin x$ increases on $[0, \pi/2]$ then *decreases* back to $0$ on $[\pi/2, \pi]$. Monotonicity fails, so it is **not a CDF**. (Restricting to $[0, \pi/2]$ would work: $F(x) = \sin x$ there is a valid CDF.)

**Step 3 — (c).** As written $H(\infty) = 1 + \frac{1}{2} = \frac{3}{2} \ne 1$; normalize by $\frac{2}{3}$:

$$
\tilde{H}(x) = \tfrac{2}{3}\left(1 - e^{-x}\right)\mathbf{1}_{x \ge 0} + \tfrac{1}{3}\mathbf{1}_{x \ge 1}.
$$

This is non-decreasing, right-continuous with a jump of height $\frac{1}{3}$ at $x = 1$, and has limits $0$ and $1$ — a **valid mixed CDF**: continuous part of weight $\frac{2}{3}$, atom of mass $\frac{1}{3}$ at $x = 1$.

$$
\boxed{\text{(a) valid; (b) invalid (not monotone); (c) valid after normalization — a mixed law}}
$$

**Key takeaway** — Check the three CDF axioms; jumps are allowed and encode atoms, but a single downward step disqualifies a function outright.

In [3]:
xs = np.linspace(-8, 8, 4001)
logistic = 1 / (1 + np.exp(-xs))
print(f"(a) monotone: {np.all(np.diff(logistic) >= 0)}, limits {logistic[0]:.2e} -> {logistic[-1]:.6f}")

t = np.linspace(0, np.pi, 4001)
print(f"(b) sin monotone on [0, pi]: {np.all(np.diff(np.sin(t)) >= -1e-15)}")

H = lambda x: (2 / 3) * (1 - np.exp(-np.clip(x, 0, None))) * (x >= 0) + (1 / 3) * (x >= 1)
grid = np.linspace(-1, 40, 40001)
vals = H(grid)
jump = H(np.array([1.0]))[0] - H(np.array([1.0 - 1e-12]))[0]
print(f"(c) monotone: {np.all(np.diff(vals) >= -1e-15)}, H(inf) -> {H(np.array([60.0]))[0]:.6f}, jump at 1 = {jump:.6f}")
assert np.all(np.diff(logistic) >= 0)
assert not np.all(np.diff(np.sin(t)) >= -1e-15)
assert abs(jump - 1 / 3) < 1e-6

(a) monotone: True, limits 3.35e-04 -> 0.999665
(b) sin monotone on [0, pi]: False
(c) monotone: True, H(inf) -> 1.000000, jump at 1 = 0.333333


### Problem L0.3 — Probability of Hitting an Exact Value

**Statement**

For $X \sim \text{Exponential}(\lambda)$ with $\lambda = 1$, compute $P(X = 2)$ and $P(1.9 \le X \le 2.1)$, and explain how both answers coexist.

**Intuition**

A continuous CDF has no jumps, so single points carry no mass; intervals still do, at a rate given by the density.

**Solution**

**Step 1.** The exponential law is continuous, so its CDF $F(x) = 1 - e^{-x}$ has no jumps and the jump formula of Theorem 4.1 gives

$$
P(X = 2) = F(2) - F(2^-) = 0 .
$$

**Step 2.** The interval around $2$ has positive probability:

$$
P(1.9 \le X \le 2.1) = F(2.1) - F(1.9) = e^{-1.9} - e^{-2.1} = 0.1495686 - 0.1224564 = 0.0271122 .
$$

**Step 3.** There is no tension: mass is spread continuously, so a single point carries none while an interval of length $0.2$ carries roughly $0.2 \times f(2) = 0.2 \times 0.1353353 = 0.0270671$, which matches to first order.

$$
\boxed{P(X = 2) = 0, \qquad P(1.9 \le X \le 2.1) = e^{-1.9} - e^{-2.1} \approx 0.027112}
$$

**Key takeaway** — Continuous laws assign probability to intervals via density times length in the small-interval limit; exact points are always null.

In [4]:
F = lambda x: 1 - np.exp(-x)
interval = F(2.1) - F(1.9)
approx = 0.2 * np.exp(-2.0)
print(f"P(1.9<=X<=2.1) = {interval:.7f}")
print(f"f(2)*0.2       = {approx:.7f}")
print(f"scipy          = {stats.expon.cdf(2.1) - stats.expon.cdf(1.9):.7f}")
assert abs(interval - 0.0271122) < 1e-6
assert abs(interval - (stats.expon.cdf(2.1) - stats.expon.cdf(1.9))) < 1e-14

P(1.9<=X<=2.1) = 0.0271122
f(2)*0.2       = 0.0270671
scipy          = 0.0271122


### Problem L0.4 — The Random Variable Is Not Random

**Statement**

Let $\Omega = \{1, \ldots, 6\}$ (fair die) and define $X(\omega) = (\omega - 3.5)^2$. List the distinct values of $X$ and its PMF, and explain in what sense $X$ is deterministic.

**Intuition**

The map is a fixed rule; only the input is random, and the PMF just collects preimage probabilities.

**Solution**

**Step 1.** Evaluate the rule: $X(1) = X(6) = 6.25$, $X(2) = X(5) = 2.25$, $X(3) = X(4) = 0.25$. Every outcome gets a definite number — there is no randomness in the rule itself.

**Step 2.** The induced law (Definition 3.2) collects preimage probabilities:

$$
p_X(0.25) = P(\{3,4\}) = \tfrac{2}{6} = \tfrac{1}{3}, \qquad p_X(2.25) = \tfrac{1}{3}, \qquad p_X(6.25) = \tfrac{1}{3}.
$$

**Step 3.** The randomness resides entirely in *which* $\omega$ the die delivers; $X$ merely transports it. This is why two functions on the same space can be dependent or independent — dependence is a property of joint preimages, not of the functions in isolation.

$$
\boxed{p_X(0.25) = p_X(2.25) = p_X(6.25) = \tfrac{1}{3}}
$$

**Key takeaway** — A random variable is a deterministic measurement rule; its distribution is the pushforward of the underlying measure through that rule.

In [5]:
omega = np.arange(1, 7)
vals = (omega - 3.5) ** 2
labels, counts = np.unique(np.round(vals, 10), return_counts=True)
pmf = counts / 6
print("values:", vals)
print("PMF   :", dict(zip(labels, pmf)))
assert np.allclose(pmf, 1 / 3) and abs(pmf.sum() - 1) < 1e-15

values: [6.25 2.25 0.25 0.25 2.25 6.25]
PMF   : {np.float64(0.25): np.float64(0.3333333333333333), np.float64(2.25): np.float64(0.3333333333333333), np.float64(6.25): np.float64(0.3333333333333333)}


## L1 — Foundations

### Problem L1.1 — From PDF to CDF and Quantiles

**Statement**

Let $f(x) = 3x^2$ on $[0,1]$ and zero elsewhere. Verify it is a density, find the CDF, the median, and $P(0.5 \lt X \le 0.9)$.

**Intuition**

Integrate once to get $F$; after that every probability and quantile question is algebra on $F$.

**Solution**

**Step 1 — density check.** $f \ge 0$ and $\int_0^1 3x^2\,dx = x^3\big\vert_0^1 = 1$.

**Step 2 — CDF.** For $x \in [0,1]$,

$$
F(x) = \int_0^x 3t^2\,dt = x^3,
$$

with $F = 0$ below $0$ and $F = 1$ above $1$.

**Step 3 — median.** Solve $F(m) = \tfrac12$: $m^3 = \tfrac12$, so $m = 2^{-1/3} = 0.7937005$. The mass piles up near $1$, dragging the median well above $0.5$.

**Step 4 — interval.**

$$
P(0.5 \lt X \le 0.9) = F(0.9) - F(0.5) = 0.729 - 0.125 = 0.604 .
$$

$$
\boxed{F(x) = x^3 \text{ on } [0,1], \quad \text{median} = 2^{-1/3} \approx 0.793700, \quad P(0.5 \lt X \le 0.9) = 0.604}
$$

**Key takeaway** — Integrate once to get the CDF, then all probability and quantile queries are pure algebra on $F$.

In [6]:
f = lambda x: 3 * x ** 2
mass, _ = integrate.quad(f, 0, 1)
median = 2 ** (-1 / 3)
interval, _ = integrate.quad(f, 0.5, 0.9)
print(f"total mass = {mass:.12f}")
print(f"median     = {median:.7f}   (F(median) = {median ** 3:.7f})")
print(f"P(0.5<X<=0.9) = {interval:.7f}")
assert abs(mass - 1) < 1e-12 and abs(median ** 3 - 0.5) < 1e-14 and abs(interval - 0.604) < 1e-12

total mass = 1.000000000000
median     = 0.7937005   (F(median) = 0.5000000)
P(0.5<X<=0.9) = 0.6040000


### Problem L1.2 — A Staircase CDF Read in Reverse

**Statement**

$X$ has CDF $F(x) = 0$ for $x \lt 0$; $F(x) = 0.3$ for $0 \le x \lt 2$; $F(x) = 0.8$ for $2 \le x \lt 5$; $F(x) = 1$ for $x \ge 5$. Recover the PMF and compute $P(X \le 2)$, $P(X \lt 2)$ and $P(X \ge 2)$.

**Intuition**

Atoms sit at the jumps with mass equal to the jump heights; then only endpoint bookkeeping remains.

**Solution**

**Step 1 — atoms.** By the jump formula of Theorem 4.1,

$$
p(0) = 0.3 - 0 = 0.3, \qquad p(2) = 0.8 - 0.3 = 0.5, \qquad p(5) = 1 - 0.8 = 0.2 .
$$

**Step 2 — endpoints.** Right-continuity means $F$ evaluates $\le$, and $F(x^-)$ evaluates $\lt$:

- $P(X \le 2) = F(2) = 0.8$ (the jump at $2$ is included);
- $P(X \lt 2) = F(2^-) = 0.3$;
- $P(X \ge 2) = 1 - P(X \lt 2) = 0.7$.

$$
\boxed{p(0) = 0.3, \; p(2) = 0.5, \; p(5) = 0.2; \quad P(X \le 2) = 0.8, \; P(X \lt 2) = 0.3, \; P(X \ge 2) = 0.7}
$$

**Key takeaway** — For discrete laws the strictness of an inequality matters exactly at atoms; $F$ answers $\le$ and $F(x^-)$ answers $\lt$.

In [7]:
atoms = np.array([0.0, 2.0, 5.0])
cum = np.array([0.3, 0.8, 1.0])
pmf = np.diff(np.concatenate(([0.0], cum)))
F = lambda x: cum[np.searchsorted(atoms, x, side="right") - 1] if x >= atoms[0] else 0.0
print("PMF:", pmf)
print(f"P(X<=2) = {F(2.0):.1f}   P(X<2) = {F(2 - 1e-9):.1f}   P(X>=2) = {1 - F(2 - 1e-9):.1f}")
assert np.allclose(pmf, [0.3, 0.5, 0.2]) and abs(pmf.sum() - 1) < 1e-15
assert abs(F(2.0) - 0.8) < 1e-12 and abs(F(2 - 1e-9) - 0.3) < 1e-12

PMF: [0.3 0.5 0.2]
P(X<=2) = 0.8   P(X<2) = 0.3   P(X>=2) = 0.7


### Problem L1.3 — Linear Change of Variables: Standardizing a Gaussian

**Statement**

Let $X \sim \mathcal{N}(\mu, \sigma^2)$ with density $f_X(x) = \frac{1}{\sigma\sqrt{2\pi}}e^{-(x-\mu)^2/(2\sigma^2)}$. Derive the density of $Z = (X - \mu)/\sigma$ and identify the distribution.

**Intuition**

An affine map has a constant Jacobian, and here that constant exactly cancels the $\sigma$ in the normalizing factor.

**Solution**

**Step 1.** The map $g(x) = (x - \mu)/\sigma$ is strictly increasing with inverse $g^{-1}(z) = \sigma z + \mu$ and $\frac{d}{dz}g^{-1}(z) = \sigma$.

**Step 2.** Apply Theorem 4.4:

$$
f_Z(z) = f_X(\sigma z + \mu)\,\lvert \sigma \rvert = \frac{1}{\sigma\sqrt{2\pi}}\exp\left(-\frac{(\sigma z)^2}{2\sigma^2}\right)\sigma = \frac{1}{\sqrt{2\pi}}e^{-z^2/2}.
$$

**Step 3.** This is $\mathcal{N}(0,1)$: the Jacobian $\sigma$ cancels the $\sigma$ in the normalizing constant. Read backwards, $X = \mu + \sigma Z$ regenerates the general Gaussian from standard noise — the reparameterization used throughout machine learning.

$$
\boxed{Z = \frac{X - \mu}{\sigma} \sim \mathcal{N}(0, 1)}
$$

**Key takeaway** — Location-scale families are one change of variables away from their standard member, and the Jacobian keeps normalization exact.

In [8]:
mu, sigma = -1.3, 2.7
zs = np.linspace(-4, 4, 9)
lhs = stats.norm(loc=mu, scale=sigma).pdf(sigma * zs + mu) * sigma
rhs = stats.norm.pdf(zs)
print(f"max |f_X(sigma z + mu)*sigma - phi(z)| = {np.max(np.abs(lhs - rhs)):.3e}")
sample = (rng.normal(mu, sigma, 200_000) - mu) / sigma
print(f"standardized sample mean/std = {sample.mean():+.4f} / {sample.std():.4f}")
assert np.max(np.abs(lhs - rhs)) < 1e-15
assert abs(sample.mean()) < 0.02 and abs(sample.std() - 1) < 0.02

max |f_X(sigma z + mu)*sigma - phi(z)| = 7.806e-18
standardized sample mean/std = +0.0001 / 1.0012


### Problem L1.4 — Inverse Transform for the Exponential

**Statement**

Derive the inverse-CDF sampler for $X \sim \text{Exponential}(\lambda)$ and compute the sample corresponding to $U = 0.2$ when $\lambda = 2$.

**Intuition**

Set $F(x) = u$ and solve for $x$; for the exponential this is one logarithm.

**Solution**

**Step 1.** The CDF is $F(x) = 1 - e^{-\lambda x}$ for $x \ge 0$. Solve $F(x) = u$:

$$
1 - e^{-\lambda x} = u \iff e^{-\lambda x} = 1 - u \iff x = -\frac{\ln(1-u)}{\lambda}.
$$

**Step 2.** By Theorem 4.3 part 1, $X = -\ln(1-U)/\lambda$ has CDF $F$. Since $1 - U \sim \text{Unif}(0,1)$ as well, the shorter form $X = -\ln U / \lambda$ is an equally valid sampler.

**Step 3.** For $U = 0.2$ and $\lambda = 2$,

$$
x = -\frac{\ln 0.8}{2} = \frac{0.2231436}{2} = 0.1115718 .
$$

(The short form would return $-\ln 0.2 / 2 = 0.8047190$ — a different number from the same $U$, but the same distribution.)

$$
\boxed{X = -\frac{\ln(1-U)}{\lambda}; \qquad U = 0.2, \; \lambda = 2 \implies x = 0.1115718}
$$

**Key takeaway** — Any law with a closed-form quantile is one algebraic step away from uniform bits; this is the standard exponential sampler in every scientific library.

In [9]:
lam = 2.0
x_at_02 = -np.log(1 - 0.2) / lam
print(f"F^-1(0.2) = {x_at_02:.7f}   short form = {-np.log(0.2) / lam:.7f}")
print(f"scipy ppf = {stats.expon(scale=1 / lam).ppf(0.2):.7f}")
sample = -np.log1p(-rng.uniform(size=200_000)) / lam
print(f"KS test vs Exp(2): p = {stats.kstest(sample, 'expon', args=(0, 1 / lam)).pvalue:.3f}")
assert abs(x_at_02 - 0.1115718) < 1e-6
assert abs(x_at_02 - stats.expon(scale=1 / lam).ppf(0.2)) < 1e-14
assert stats.kstest(sample, "expon", args=(0, 1 / lam)).pvalue > 0.01

F^-1(0.2) = 0.1115718   short form = 0.8047190
scipy ppf = 0.1115718
KS test vs Exp(2): p = 0.554


### Problem L1.5 — Non-Monotone Transform: Square of a Uniform

**Statement**

Let $X \sim \text{Unif}(-1, 2)$. Find the density of $Y = X^2$.

**Intuition**

Two branches $\pm\sqrt{y}$ contribute while both stay inside $[-1,2]$; past $y = 1$ the negative branch leaves the support.

**Solution**

**Step 1.** $f_X = \frac{1}{3}$ on $[-1, 2]$, and the support of $Y$ is $[0,4]$. Work through the CDF, splitting where the branch structure changes.

**Step 2 — case $0 \le y \le 1$.** Both roots lie in the support:

$$
F_Y(y) = P\left(-\sqrt{y} \le X \le \sqrt{y}\right) = \frac{2\sqrt{y}}{3} \implies f_Y(y) = \frac{1}{3\sqrt{y}} .
$$

**Step 3 — case $1 \lt y \le 4$.** Only the positive root is inside $[-1,2]$, since $-\sqrt{y} \lt -1$:

$$
F_Y(y) = P\left(-1 \le X \le \sqrt{y}\right) = \frac{\sqrt{y} + 1}{3} \implies f_Y(y) = \frac{1}{6\sqrt{y}} .
$$

**Step 4 — normalization check.**

$$
\int_0^1 \frac{dy}{3\sqrt{y}} + \int_1^4 \frac{dy}{6\sqrt{y}} = \frac{2}{3} + \frac{2\sqrt{y}}{6}\bigg\vert_1^4 = \frac{2}{3} + \frac{4-2}{6} = 1 .
$$

$$
\boxed{f_Y(y) = \frac{1}{3\sqrt{y}} \text{ on } (0,1], \qquad f_Y(y) = \frac{1}{6\sqrt{y}} \text{ on } (1,4]}
$$

**Key takeaway** — For non-monotone maps, count contributing branches region by region: the density formula changes wherever a branch exits the support.

In [10]:
f_Y = lambda y: np.where(y <= 1, 1 / (3 * np.sqrt(y)), 1 / (6 * np.sqrt(y)))
mass, _ = integrate.quad(f_Y, 0, 4, points=[1.0])
y_sim = rng.uniform(-1, 2, 400_000) ** 2
F_Y = lambda y: np.where(y <= 1, 2 * np.sqrt(y) / 3, (np.sqrt(y) + 1) / 3)
emp = np.array([np.mean(y_sim <= t) for t in [0.25, 1.0, 2.25, 4.0]])
thy = F_Y(np.array([0.25, 1.0, 2.25, 4.0]))
print(f"total mass = {mass:.12f}")
print(f"empirical CDF {emp}")
print(f"theoretical   {thy}")
assert abs(mass - 1) < 1e-9 and np.max(np.abs(emp - thy)) < 5e-3

total mass = 1.000000000000
empirical CDF [0.3338 0.665  0.8326 1.    ]
theoretical   [0.3333 0.6667 0.8333 1.    ]


### Problem L1.6 — Mixed Distribution: Rainfall Model

**Statement**

Daily rainfall is $0$ with probability $0.6$; given rain, the amount is $\text{Exponential}(1/10)$ (mean $10$ mm). Write the CDF of rainfall $R$ and compute $P(R \le 5)$ and $P(R = 0)$.

**Intuition**

An atom and a continuous part live in the same CDF: a jump at $0$ followed by a smooth climb.

**Solution**

**Step 1.** $R$ mixes an atom at $0$ of weight $0.6$ with a continuous part of weight $0.4$:

$$
F_R(r) = 0.6 + 0.4\left(1 - e^{-r/10}\right) \text{ for } r \ge 0, \qquad F_R(r) = 0 \text{ for } r \lt 0.
$$

**Step 2.** The jump at $0$ has height $F_R(0) - F_R(0^-) = 0.6 - 0$, so $P(R = 0) = 0.6$.

**Step 3.**

$$
P(R \le 5) = 0.6 + 0.4\left(1 - e^{-0.5}\right) = 0.6 + 0.4 \times 0.3934693 = 0.7573877 .
$$

**Step 4.** Neither a PMF nor a PDF alone describes $R$, but the CDF handles the mixture effortlessly — the $(a,b,c) = (0.6, 0.4, 0)$ corner of Theorem 4.2. Zero-inflated and hurdle models, and tobit/censored regression likelihoods, have exactly this structure.

$$
\boxed{P(R = 0) = 0.6, \qquad P(R \le 5) = 0.6 + 0.4\left(1 - e^{-1/2}\right) = 0.7573877}
$$

**Key takeaway** — Mixed laws are atoms plus continuous mass, and the CDF is the only universal container for both.

In [11]:
F_R = lambda r: np.where(r < 0, 0.0, 0.6 + 0.4 * (1 - np.exp(-np.clip(r, 0, None) / 10)))
p5 = F_R(np.array([5.0]))[0]
atom = F_R(np.array([0.0]))[0] - F_R(np.array([-1e-12]))[0]
rain = np.where(rng.uniform(size=400_000) < 0.6, 0.0, rng.exponential(10.0, 400_000))
print(f"P(R=0)  exact {atom:.7f}   simulated {np.mean(rain == 0):.4f}")
print(f"P(R<=5) exact {p5:.7f}   simulated {np.mean(rain <= 5):.4f}")
assert abs(p5 - 0.7573877) < 1e-6 and abs(atom - 0.6) < 1e-12
assert abs(np.mean(rain <= 5) - p5) < 5e-3

P(R=0)  exact 0.6000000   simulated 0.6004
P(R<=5) exact 0.7573877   simulated 0.7573


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Reparameterization Trick in VAEs

**Statement**

A variational autoencoder needs gradients of $E_{z \sim \mathcal{N}(\mu, \sigma^2)}[\ell(z)]$ with respect to $\mu$ and $\sigma$. Show how writing $z = \mu + \sigma\varepsilon$ with $\varepsilon \sim \mathcal{N}(0,1)$ turns the sampling node into a differentiable transform, and derive $\partial z/\partial\mu$ and $\partial z/\partial\sigma$.

**Intuition**

Move the parameters out of the sampling operation and into a deterministic map, so the chain rule can reach them.

**Solution**

**Step 1.** By Problem L1.3 read in reverse, $\mu + \sigma\varepsilon \sim \mathcal{N}(\mu, \sigma^2)$ — the same distribution, with the randomness isolated in parameter-free noise. Hence

$$
E_{z \sim \mathcal{N}(\mu,\sigma^2)}[\ell(z)] = E_{\varepsilon \sim \mathcal{N}(0,1)}\left[\ell(\mu + \sigma\varepsilon)\right].
$$

**Step 2.** The right-hand expectation has a parameter-free measure, so (dominated convergence permitting) the gradient passes inside:

$$
\frac{\partial}{\partial\mu}E[\ell] = E\left[\ell'(z)\right], \qquad \frac{\partial}{\partial\sigma}E[\ell] = E\left[\ell'(z)\,\varepsilon\right],
$$

using $\partial z/\partial\mu = 1$ and $\partial z/\partial\sigma = \varepsilon$.

**Step 3.** A single $\varepsilon$ per data point gives an unbiased, low-variance gradient estimate — impossible if $z$ were sampled directly, since the parameters would sit inside the sampling operation rather than in a differentiable map.

$$
\boxed{z = \mu + \sigma\varepsilon \implies \partial_\mu z = 1, \; \partial_\sigma z = \varepsilon; \text{ gradients flow through the transform}}
$$

**Key takeaway** — Expressing a random variable as a deterministic transform of fixed noise converts "differentiate through sampling" into ordinary backpropagation.

In [12]:
# ell(z) = z^2, so E[ell] = mu^2 + sigma^2 with gradients (2 mu, 2 sigma).
mu, sigma, m = 0.7, 1.3, 2_000_000
eps = rng.standard_normal(m)
z = mu + sigma * eps
grad_mu = np.mean(2 * z * 1.0)
grad_sigma = np.mean(2 * z * eps)
print(f"reparameterized gradient  d/dmu    = {grad_mu:.4f}   exact {2 * mu:.4f}")
print(f"reparameterized gradient  d/dsigma = {grad_sigma:.4f}   exact {2 * sigma:.4f}")
print(f"finite difference d/dsigma          = {((mu**2 + (sigma+1e-5)**2) - (mu**2 + (sigma-1e-5)**2)) / 2e-5:.4f}")
assert abs(grad_mu - 2 * mu) < 0.02 and abs(grad_sigma - 2 * sigma) < 0.02

reparameterized gradient  d/dmu    = 1.3977   exact 1.4000
reparameterized gradient  d/dsigma = 2.5962   exact 2.6000
finite difference d/dsigma          = 2.6000


### Problem L2.2 — Normalizing Flow Log-Density

**Statement**

A one-dimensional flow maps latent $z \sim \mathcal{N}(0,1)$ through $x = g(z) = az + \tanh z$ with $a \gt 0$. Write the exact log-density $\ln p_X(x)$ in terms of $z = g^{-1}(x)$, explain why $g$ is invertible, and say what changes in higher dimensions.

**Intuition**

The log-density of the output is the log-density of the latent minus the log stretch factor of the map.

**Solution**

**Step 1 — invertibility.** $g'(z) = a + \operatorname{sech}^2 z \gt 0$ for all $z$, so $g$ is strictly increasing, hence invertible; the inverse is computed numerically, for example by bisection or Newton's method.

**Step 2 — log-density.** Theorem 4.4 in log form gives

$$
\ln p_X(x) = \ln p_Z(z) - \ln\left\lvert g'(z)\right\rvert = -\frac{z^2}{2} - \frac{1}{2}\ln(2\pi) - \ln\left(a + \operatorname{sech}^2 z\right), \qquad z = g^{-1}(x).
$$

**Step 3 — sign.** Where $g$ *stretches* space ($g'$ large) the density is *diluted*, so the log-density falls. That single minus sign is the entire content of the Jacobian term.

**Step 4 — higher dimensions.** By Theorem 4.5,

$$
\ln p_X(x) = \ln p_Z\left(g^{-1}(x)\right) - \ln\left\lvert \det J_g\left(g^{-1}(x)\right)\right\rvert,
$$

and flow architectures (coupling layers, autoregressive flows) are engineered so $\det J_g$ is triangular and costs $O(d)$ rather than $O(d^3)$.

$$
\boxed{\ln p_X(x) = \ln p_Z(z) - \ln\left(a + \operatorname{sech}^2 z\right), \quad z = g^{-1}(x)}
$$

**Key takeaway** — A normalizing flow is the change-of-variables theorem promoted to a trainable architecture; the log-Jacobian is the price of reshaping distributions.

In [13]:
a = 0.8
g = lambda z: a * z + np.tanh(z)
gp = lambda z: a + 1 / np.cosh(z) ** 2

def g_inv(x, iters=200):
    lo, hi = np.full_like(x, -60.0), np.full_like(x, 60.0)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        take = g(mid) < x
        lo, hi = np.where(take, mid, lo), np.where(take, hi, mid)
    return 0.5 * (lo + hi)

log_p_X = lambda x: stats.norm.logpdf(g_inv(x)) - np.log(gp(g_inv(x)))
mass, _ = integrate.quad(lambda x: np.exp(log_p_X(np.array([x]))[0]), -50, 50)
x_sim = g(rng.standard_normal(200_000))
print(f"integral of p_X = {mass:.10f}")
print(f"log p_X(0.5) = {log_p_X(np.array([0.5]))[0]:.6f}")
print(f"P(X<=0.5): model {integrate.quad(lambda x: np.exp(log_p_X(np.array([x]))[0]), -50, 0.5)[0]:.4f}  simulated {np.mean(x_sim <= 0.5):.4f}")
assert abs(mass - 1) < 1e-6

integral of p_X = 1.0000000000
log p_X(0.5) = -1.503646
P(X<=0.5): model 0.6109  simulated 0.6109


### Problem L2.3 — PIT Calibration of a Forecaster

**Statement**

A weather model issues predictive CDFs $F_t$ for daily temperature; over $n$ days you record PIT values $u_t = F_t(y_t)$ of the realized temperatures $y_t$. (a) What distribution should the $u_t$ have if the forecasts are perfect? (b) The histogram of $u_t$ is U-shaped. Diagnose the model. (c) It is hump-shaped. Diagnose.

**Intuition**

Theorem 4.3 part 2 says a correct predictive CDF flattens its own outcomes into uniform noise, so any bump is a modelling error made visible.

**Solution**

**Step 1 — (a).** By the probability integral transform, if $y_t \sim F_t$ and $F_t$ is continuous then $u_t \sim \text{Unif}(0,1)$, independent across days: the histogram should be flat.

**Step 2 — (b).** A U-shape means too many realizations land in the extreme quantiles of the forecast — reality escapes the predicted range too often. The predictive distributions are **too narrow (overconfident)**: forecast variance underestimates true variability.

**Step 3 — (c).** A hump means realizations cluster in the middle quantiles, so the forecast spreads mass too widely: **too wide (underconfident)**.

**Step 4.** A one-sided slope (mass near $1$) instead indicates bias — reality systematically exceeds the forecast median.

$$
\boxed{\text{Perfect} \Rightarrow \text{Unif}(0,1); \; \text{U-shape} \Rightarrow \text{overconfident}; \; \text{hump} \Rightarrow \text{underconfident}}
$$

**Key takeaway** — The PIT turns "is my predictive distribution right?" into "is this histogram flat?", the canonical calibration diagnostic in probabilistic forecasting and uncertainty-aware machine learning.

In [14]:
y_real = rng.normal(0.0, 1.0, 40_000)
for name, scale in [("correct  1.0", 1.0), ("narrow   0.5", 0.5), ("wide     2.0", 2.0)]:
    pit = stats.norm(0.0, scale).cdf(y_real)
    edge = np.mean((pit < 0.1) | (pit > 0.9))       # extreme decile mass, 0.2 if uniform
    mid = np.mean((pit > 0.4) & (pit < 0.6))        # central mass, 0.2 if uniform
    print(f"{name}: KS vs Unif = {stats.kstest(pit, 'uniform').statistic:.3f}, "
          f"tail mass = {edge:.3f}, central mass = {mid:.3f}")
pit_ok = stats.norm(0, 1.0).cdf(y_real)
pit_narrow = stats.norm(0, 0.5).cdf(y_real)
pit_wide = stats.norm(0, 2.0).cdf(y_real)
assert stats.kstest(pit_ok, "uniform").statistic < 0.01
assert np.mean((pit_narrow < 0.1) | (pit_narrow > 0.9)) > 0.3      # U-shape
assert np.mean((pit_wide > 0.4) & (pit_wide < 0.6)) > 0.3          # hump

correct  1.0: KS vs Unif = 0.005, tail mass = 0.201, central mass = 0.200
narrow   0.5: KS vs Unif = 0.166, tail mass = 0.527, central mass = 0.098
wide     2.0: KS vs Unif = 0.161, tail mass = 0.011, central mass = 0.386


### Problem L2.4 — Free Path Lengths in Particle Transport

**Statement**

A photon travelling through a homogeneous medium with attenuation coefficient $\mu_a$ survives distance $s$ without interaction with probability $e^{-\mu_a s}$ (Beer–Lambert). (a) Derive the PDF of the free path length $S$. (b) Give the inverse-transform sampler used in Monte Carlo transport codes. (c) Compute the median free path for $\mu_a = 0.2\ \text{cm}^{-1}$.

**Intuition**

Beer–Lambert *is* a survival function, so the law is exponential and everything follows from $S(s) = 1 - F(s)$.

**Solution**

**Step 1 — (a).** The survival probability is the complementary CDF: $P(S \gt s) = e^{-\mu_a s}$, so

$$
F_S(s) = 1 - e^{-\mu_a s}, \qquad f_S(s) = \mu_a e^{-\mu_a s}, \quad s \ge 0,
$$

exponential with rate $\mu_a$ and mean free path $1/\mu_a$. Equivalently, the hazard rate of Definition 3.8 is the constant $\mu_a$.

**Step 2 — (b).** Inverse transform: $s = -\ln(1-u)/\mu_a$, in practice $s = -\ln u/\mu_a$ with $u \sim \text{Unif}(0,1)$. This two-operation sampler runs trillions of times in reactor-shielding and medical-dosimetry codes.

**Step 3 — (c).** Solve $F_S(m) = \tfrac12$:

$$
m = \frac{\ln 2}{\mu_a} = \frac{0.6931472}{0.2} = 3.4657359 \text{ cm},
$$

notably *less* than the mean free path $1/\mu_a = 5$ cm: the right-skewed exponential puts its median below its mean.

$$
\boxed{f_S(s) = \mu_a e^{-\mu_a s}, \qquad s = -\ln u/\mu_a, \qquad m = \ln 2/\mu_a = 3.4657 \text{ cm}}
$$

**Key takeaway** — Physical attenuation laws are survival functions in disguise, and Monte Carlo transport is inverse-transform sampling applied to physics.

In [15]:
mu_a = 0.2
median = np.log(2) / mu_a
s = -np.log1p(-rng.uniform(size=400_000)) / mu_a
print(f"median free path = {median:.7f} cm   mean = {1 / mu_a:.4f} cm")
print(f"simulated median = {np.median(s):.4f} cm   simulated mean = {s.mean():.4f} cm")
print(f"scipy ppf(0.5)   = {stats.expon(scale=1 / mu_a).ppf(0.5):.7f} cm")
assert abs(median - 3.4657359) < 1e-6
assert abs(np.median(s) - median) < 0.05 and median < 1 / mu_a

median free path = 3.4657359 cm   mean = 5.0000 cm
simulated median = 3.4571 cm   simulated mean = 4.9876 cm
scipy ppf(0.5)   = 3.4657359 cm


### Problem L2.5 — Quantiles for Value-at-Risk and Distributional RL

**Statement**

Portfolio loss $L \sim \mathcal{N}(\mu = 1, \sigma^2 = 25)$ (in units of $10^3$ USD). (a) Compute $\text{VaR}_{0.99} = F_L^{-1}(0.99)$ using $z_{0.99} = 2.3263$. (b) In distributional RL (QR-DQN) a network predicts quantiles $\theta_i = F^{-1}(\tau_i)$ at levels $\tau_i = \frac{2i-1}{2N}$. For $N = 4$ list the levels, and explain why learning quantiles rather than the mean matters in both settings.

**Intuition**

Gaussian quantiles are affine in the standard-normal quantiles, and quantiles keep exactly the tail information an expectation throws away.

**Solution**

**Step 1 — (a).** Quantiles of a location-scale family transform affinely (Problem L1.3):

$$
\text{VaR}_{0.99} = \mu + \sigma z_{0.99} = 1 + 5 \times 2.3263 = 12.6315 .
$$

With probability $99\%$ the loss does not exceed about \$12.63k; the worst $1\%$ of scenarios exceed it.

**Step 2 — (b).** For $N = 4$, $\tau_i = \frac{2i-1}{8}$ gives

$$
\tau \in \{0.125,\; 0.375,\; 0.625,\; 0.875\},
$$

the midpoints of four equal-probability bins, which is the optimal $4$-point approximation of the return distribution under the pinball loss.

**Step 3.** Means hide tail structure: two policies with equal expected return can differ enormously in downside risk, and two portfolios with equal mean loss can have very different VaR. Quantile representations retain exactly the information expectation integrates away.

$$
\boxed{\text{VaR}_{0.99} = 12.6315; \qquad \tau = \{0.125, 0.375, 0.625, 0.875\}}
$$

**Key takeaway** — The quantile function is the risk-aware face of a distribution: finance (VaR) and distributional RL (QR-DQN) both learn $F^{-1}$, not just $E[X]$.

In [16]:
mu, sigma = 1.0, 5.0
var99_hand = mu + sigma * 2.3263
var99_exact = stats.norm(mu, sigma).ppf(0.99)
taus = (2 * np.arange(1, 5) - 1) / 8
print(f"VaR_0.99 with z=2.3263 : {var99_hand:.4f}")
print(f"VaR_0.99 exact (scipy) : {var99_exact:.4f}")
print(f"QR-DQN levels N=4      : {taus}")
print(f"quantiles at those tau : {stats.norm(mu, sigma).ppf(taus)}")
assert abs(var99_hand - 12.6315) < 1e-4 and abs(var99_hand - var99_exact) < 0.002
assert np.allclose(taus, [0.125, 0.375, 0.625, 0.875])

VaR_0.99 with z=2.3263 : 12.6315
VaR_0.99 exact (scipy) : 12.6317
QR-DQN levels N=4      : [0.125 0.375 0.625 0.875]
quantiles at those tau : [-4.7517 -0.5932  2.5932  6.7517]


### Problem L2.6 — Softmax Temperature as a Pushforward

**Statement**

A language model produces logits $(2.0, 1.0, 0.0)$ over three tokens. Sampling uses temperature $T$: probabilities $p_i \propto e^{z_i/T}$. Compute the token distributions for $T = 1$ and $T = 0.5$, describe the induced random variable, and give the limiting laws as $T \to 0^+$ and $T \to \infty$.

**Intuition**

Temperature rescales the logit gaps, and the softmax pushes those gaps onto a PMF that interpolates between a point mass and the uniform law.

**Solution**

**Step 1 — $T = 1$.** Weights $(e^2, e^1, e^0) = (7.3890561, 2.7182818, 1)$ with sum $11.1073379$:

$$
p = (0.6652410,\; 0.2447285,\; 0.0900306).
$$

**Step 2 — $T = 0.5$.** The logits double: weights $(e^4, e^2, e^0) = (54.5981500, 7.3890561, 1)$ with sum $62.9872061$:

$$
p = (0.8668133,\; 0.1173104,\; 0.0158762).
$$

**Step 3.** The sampled token is a discrete random variable $X_T$ on $\{1,2,3\}$ whose PMF is the softmax pushforward of the logits; $T$ reshapes it monotonically in the gaps $z_i - z_j$.

**Step 4 — limits.** As $T \to 0^+$ the mass concentrates on the argmax, giving a point mass at token $1$ (greedy decoding). As $T \to \infty$ the exponents flatten to the uniform law $\left(\tfrac13,\tfrac13,\tfrac13\right)$.

$$
\boxed{T = 1: (0.66524, 0.24473, 0.09003); \quad T = 0.5: (0.86681, 0.11731, 0.01588)}
$$

**Key takeaway** — Decoding hyperparameters are transformations of a PMF; temperature interpolates between the point mass and the uniform law while preserving the ranking of outcomes.

In [17]:
logits = np.array([2.0, 1.0, 0.0])
def softmax(z, T):
    e = np.exp((z - z.max()) / T)      # shift for numerical stability
    return e / e.sum()
p1, p05 = softmax(logits, 1.0), softmax(logits, 0.5)
print(f"T = 1.0 : {p1}")
print(f"T = 0.5 : {p05}")
print(f"T -> 0  : {softmax(logits, 1e-3)}")
print(f"T -> inf: {softmax(logits, 1e6)}")
assert np.allclose(p1, [0.66524096, 0.24472847, 0.09003057], atol=1e-8)
assert np.allclose(p05, [0.86681333, 0.11731043, 0.01587624], atol=1e-8)
assert np.allclose(softmax(logits, 1e-3), [1, 0, 0], atol=1e-6)
assert np.allclose(softmax(logits, 1e6), [1 / 3, 1 / 3, 1 / 3], atol=1e-5)

T = 1.0 : [0.6652 0.2447 0.09  ]
T = 0.5 : [0.8668 0.1173 0.0159]
T -> 0  : [1. 0. 0.]
T -> inf: [0.3333 0.3333 0.3333]


## L3 — Challenge Proofs

### Problem L3.1 — Cauchy from a Uniform Angle: a Pathological Pushforward

**Statement**

A spinner at the origin picks an angle $\Theta \sim \text{Unif}(-\pi/2, \pi/2)$ and shoots a ray hitting the vertical wall $x = 1$ at height $Y = \tan\Theta$. Derive the density of $Y$ and show it has no mean.

**Intuition**

The tangent map stretches the ends of the angle interval without bound, so a bounded uniform turns into a heavy-tailed law.

**Solution**

**Step 1.** $g(\theta) = \tan\theta$ is strictly increasing on $(-\pi/2, \pi/2)$ with inverse $\theta = \arctan y$ and $\frac{d\theta}{dy} = \frac{1}{1+y^2}$. With $f_\Theta = \frac{1}{\pi}$, Theorem 4.4 gives

$$
f_Y(y) = f_\Theta(\arctan y)\left\lvert \frac{d\theta}{dy}\right\rvert = \frac{1}{\pi\left(1+y^2\right)}, \qquad y \in \mathbb{R},
$$

the standard Cauchy density.

**Step 2.** Test absolute integrability, which is what the definition of $E[Y]$ requires:

$$
\int_{-\infty}^{\infty} \frac{\lvert y \rvert}{\pi\left(1+y^2\right)}\,dy = \frac{2}{\pi}\int_0^{\infty}\frac{y}{1+y^2}\,dy = \frac{1}{\pi}\ln\left(1+y^2\right)\bigg\vert_0^{\infty} = \infty .
$$

**Step 3.** The integral diverges, so $E[Y]$ does not exist — not even as $\pm\infty$, and the symmetric principal value $0$ is not a mean. Sample means of Cauchy data therefore do not converge; the integrability hypothesis of the law of large numbers fails.

$$
\boxed{f_Y(y) = \frac{1}{\pi\left(1+y^2\right)}; \qquad E[Y] \text{ does not exist}}
$$

**Key takeaway** — A perfectly innocent transformation of a bounded uniform variable produces a law with no mean: heavy tails are one Jacobian away.

In [18]:
theta = rng.uniform(-np.pi / 2, np.pi / 2, 200_000)
y = np.tan(theta)
grid = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])
print(f"pdf  formula {1 / (np.pi * (1 + grid ** 2))}")
print(f"pdf  scipy   {stats.cauchy.pdf(grid)}")
for T in [10, 100, 1000, 10_000]:
    val, _ = integrate.quad(lambda t: abs(t) / (np.pi * (1 + t ** 2)), -T, T)
    print(f"integral of |y| f(y) over [-{T}, {T}] = {val:.4f}   (2/pi)*ln(T) = {2 / np.pi * np.log(T):.4f}")
run = np.array([np.mean(y[:k]) for k in [10**3, 10**4, 10**5, 2 * 10**5]])
print(f"running sample means (do not settle): {run}")
assert np.allclose(1 / (np.pi * (1 + grid ** 2)), stats.cauchy.pdf(grid), atol=1e-14)

pdf  formula [0.0318 0.1592 0.3183 0.1592 0.0318]
pdf  scipy   [0.0318 0.1592 0.3183 0.1592 0.0318]
integral of |y| f(y) over [-10, 10] = 1.4690   (2/pi)*ln(T) = 1.4659
integral of |y| f(y) over [-100, 100] = 2.9318   (2/pi)*ln(T) = 2.9317
integral of |y| f(y) over [-1000, 1000] = 4.3976   (2/pi)*ln(T) = 4.3976
integral of |y| f(y) over [-10000, 10000] = 5.8635   (2/pi)*ln(T) = 5.8635
running sample means (do not settle): [  1.4923  -1.2544   1.7302 -45.5695]


### Problem L3.2 — Box–Muller Transform (Bivariate Change of Variables)

**Statement**

Show that if $U_1, U_2 \sim \text{Unif}(0,1)$ are independent then

$$
Z_1 = \sqrt{-2\ln U_1}\,\cos\left(2\pi U_2\right), \qquad Z_2 = \sqrt{-2\ln U_1}\,\sin\left(2\pi U_2\right)
$$

are independent standard normals.

**Intuition**

In polar coordinates the standard bivariate normal factors into a uniform angle and a Rayleigh radius; Box–Muller simply builds those two pieces.

**Solution**

**Step 1 — build the polar pieces.** Let $R = \sqrt{-2\ln U_1}$ and $\Theta = 2\pi U_2$. Then $\Theta \sim \text{Unif}(0, 2\pi)$, and $R^2 = -2\ln U_1 \sim \text{Exponential}(1/2)$, since $P(R^2 \gt t) = P\left(U_1 \lt e^{-t/2}\right) = e^{-t/2}$; the two are independent because $U_1$ and $U_2$ are.

**Step 2 — factor the target in polar coordinates.** For $z_1 = r\cos\theta$, $z_2 = r\sin\theta$, the Jacobian determinant of $(r,\theta)\mapsto(z_1,z_2)$ is $r$, so by Theorem 4.5

$$
f_{Z_1,Z_2}(z_1,z_2) = \frac{1}{2\pi}e^{-(z_1^2+z_2^2)/2} \iff f_{R,\Theta}(r,\theta) = \frac{1}{2\pi}e^{-r^2/2}\,r = \underbrace{\frac{1}{2\pi}}_{\Theta \text{ uniform}} \cdot \underbrace{r e^{-r^2/2}}_{\text{Rayleigh}} .
$$

**Step 3 — match.** Our constructed pair has exactly this law: $\Theta$ is uniform on $(0,2\pi)$, and $P(R \le r) = P\left(R^2 \le r^2\right) = 1 - e^{-r^2/2}$, whose derivative is $re^{-r^2/2}$ — the Rayleigh density.

**Step 4.** Since the joint densities agree, $(Z_1, Z_2)$ has the standard bivariate normal law, whose density factors; hence $Z_1$ and $Z_2$ are independent $\mathcal{N}(0,1)$. $\blacksquare$

$$
\boxed{(Z_1, Z_2) \sim \mathcal{N}(0,1) \times \mathcal{N}(0,1) \text{, independent}}
$$

**Key takeaway** — The Gaussian's rotational symmetry makes polar coordinates factor perfectly, so Box–Muller converts two uniforms into two exact Gaussians with no rejection.

In [19]:
n = 200_000
u1, u2 = rng.uniform(1e-15, 1, n), rng.uniform(0, 1, n)
r = np.sqrt(-2 * np.log(u1))
z1, z2 = r * np.cos(2 * np.pi * u2), r * np.sin(2 * np.pi * u2)
print(f"KS(z1, N(0,1)) p = {stats.kstest(z1, 'norm').pvalue:.3f}")
print(f"KS(z2, N(0,1)) p = {stats.kstest(z2, 'norm').pvalue:.3f}")
print(f"corr(z1, z2)     = {np.corrcoef(z1, z2)[0, 1]:+.4f}")
print(f"KS(R^2, Exp(1/2)) p = {stats.kstest(r ** 2, 'expon', args=(0, 2)).pvalue:.3f}")
assert stats.kstest(z1, "norm").pvalue > 0.01 and stats.kstest(z2, "norm").pvalue > 0.01
assert abs(np.corrcoef(z1, z2)[0, 1]) < 0.01

KS(z1, N(0,1)) p = 0.968
KS(z2, N(0,1)) p = 0.826
corr(z1, z2)     = +0.0003
KS(R^2, Exp(1/2)) p = 0.512


### Problem L3.3 — Skorokhod Representation on $[0,1]$ (Universality of the Uniform)

**Statement**

Prove that for any collection of CDFs $F_1, F_2, \ldots$ there exist random variables $X_1, X_2, \ldots$ *all defined on the single probability space* $\left([0,1], \mathcal{B}, \lambda\right)$ with $X_i \sim F_i$. Conclude that one uniform random number generator suffices to simulate any distribution.

**Intuition**

The quantile construction of Proof 5.1 already turns a single $\omega \in [0,1]$ into one variable with a prescribed law, and nothing forces you to use a fresh $\omega$ for each law: feed the *same* $\omega$ to every quantile function at once.

**Solution**

**Step 1 — the space.** Let $\Omega = [0,1]$ with Lebesgue measure $\lambda$ and $U(\omega) = \omega$, so $U \sim \text{Unif}(0,1)$: indeed $\lambda(\{\omega : \omega \le u\}) = u$.

**Step 2 — the variables.** For each $i$ set

$$
X_i(\omega) = F_i^{-1}(\omega) = \inf\{x : F_i(x) \ge \omega\}.
$$

Each $X_i$ is non-decreasing in $\omega$, hence Borel measurable, so it is a random variable on this one space.

**Step 3 — the law.** By the Galois inequality $F_i^{-1}(\omega) \le x \iff \omega \le F_i(x)$ established in Proof 5.1,

$$
P(X_i \le x) = \lambda\left(\{\omega : \omega \le F_i(x)\}\right) = F_i(x),
$$

so $X_i \sim F_i$ exactly — and every $X_i$ is a function of the *same* $\omega$. $\blacksquare$

**Step 4 — consequences.** (i) A single stream of uniform bits drives the simulation of arbitrarily many laws, which is the software reality of one PRNG state with many samplers. (ii) This construction is the first step of Skorokhod's representation theorem, which upgrades convergence in distribution to almost-sure convergence on a common space.

$$
\boxed{X_i = F_i^{-1}(U) \text{ on } \left([0,1], \mathcal{B}, \lambda\right) \text{ realizes any family of laws}}
$$

**Key takeaway** — The unit interval with Lebesgue measure is a universal probability space: all of simulation is deterministic functions of one uniform seed stream.

In [20]:
omega = rng.uniform(0, 1, 200_000)                    # one stream
X_exp = stats.expon(scale=1 / 1.5).ppf(omega)         # three different laws
X_norm = stats.norm(2.0, 3.0).ppf(omega)
X_binom = stats.binom(10, 0.3).ppf(omega)
print(f"Exp(1.5)   KS p = {stats.kstest(X_exp, 'expon', args=(0, 1 / 1.5)).pvalue:.3f}")
print(f"N(2,9)     KS p = {stats.kstest(X_norm, 'norm', args=(2.0, 3.0)).pvalue:.3f}")
print(f"Bin(10,.3) PMF  = {np.bincount(X_binom.astype(int), minlength=11)[:5] / omega.size}")
print(f"           exact= {stats.binom(10, 0.3).pmf(np.arange(5))}")
assert stats.kstest(X_exp, "expon", args=(0, 1 / 1.5)).pvalue > 0.01
assert stats.kstest(X_norm, "norm", args=(2.0, 3.0)).pvalue > 0.01
assert np.max(np.abs(np.bincount(X_binom.astype(int), minlength=11) / omega.size
                     - stats.binom(10, 0.3).pmf(np.arange(11)))) < 5e-3

Exp(1.5)   KS p = 0.304
N(2,9)     KS p = 0.304
Bin(10,.3) PMF  = [0.0277 0.1207 0.2352 0.2667 0.2009]
           exact= [0.0282 0.1211 0.2335 0.2668 0.2001]


### Problem L3.4 — A Continuous but Non-Smooth CDF: the Devil's Staircase

**Statement**

The Cantor distribution is defined by $X = \sum_{k=1}^{\infty} \frac{2D_k}{3^k}$ where the $D_k$ are i.i.d. $\text{Bernoulli}(1/2)$. Show that its CDF is continuous (no atoms) yet $X$ has no density, and compute $E[X]$.

**Intuition**

Every single value needs infinitely many coin flips to pin down, so no point carries mass; yet all the mass sits on a set of length zero, so no density can exist.

**Solution**

**Step 1 — no atoms.** A specific value $x$ determines its base-$3$ digit sequence $(d_k)$ with digits in $\{0,2\}$. Then

$$
P(X = x) \le P\left(D_1 = d_1/2, \ldots, D_n = d_n/2\right) = 2^{-n} \xrightarrow[n \to \infty]{} 0,
$$

so $P(X = x) = 0$ for every $x$ and the CDF $F$ is continuous.

**Step 2 — no density.** $X$ takes values in the Cantor set $C$ (base-$3$ expansions omitting the digit $1$), and $C$ has Lebesgue measure zero: at stage $n$ of its construction $C$ sits inside $2^n$ intervals of total length $(2/3)^n \to 0$. If a density $f$ existed then

$$
1 = P(X \in C) = \int_C f\,dx = 0,
$$

a contradiction. So $F$ is continuous, non-decreasing, and increases only on a null set: singular continuous in the sense of Definition 3.6, the third corner of Theorem 4.2. Its graph is the devil's staircase, flat almost everywhere yet climbing from $0$ to $1$.

**Step 3 — mean.** By linearity and $E[D_k] = \tfrac12$,

$$
E[X] = \sum_{k=1}^{\infty}\frac{2E[D_k]}{3^k} = \sum_{k=1}^{\infty}\frac{1}{3^k} = \frac{1/3}{1 - 1/3} = \frac{1}{2},
$$

also forced by the symmetry $X \overset{d}{=} 1 - X$.

$$
\boxed{F \text{ is continuous with no density (singular); } E[X] = \tfrac{1}{2}}
$$

**Key takeaway** — "Continuous CDF" and "has a density" are different properties; the Lebesgue decomposition allows a third, singular type of law beyond discrete and absolutely continuous.

In [21]:
K = 40
D = rng.integers(0, 2, size=(200_000, K))
X = (2 * D / 3.0 ** np.arange(1, K + 1)).sum(axis=1)
print(f"E[X] simulated = {X.mean():.5f}   exact = 0.5")
print(f"symmetry check: mean of 1 - X = {(1 - X).mean():.5f}")
# every value is essentially unique -> no atoms; and X avoids the removed middle third (1/3, 2/3)
print(f"largest repeated-value frequency = {np.max(np.unique(np.round(X, 9), return_counts=True)[1]) / X.size:.2e}")
print(f"P(1/3 < X < 2/3) = {np.mean((X > 1 / 3) & (X < 2 / 3)):.4f}  (Cantor set omits this gap)")
assert abs(X.mean() - 0.5) < 5e-3
assert np.mean((X > 1 / 3 + 1e-9) & (X < 2 / 3 - 1e-9)) == 0.0

E[X] simulated = 0.49933   exact = 0.5
symmetry check: mean of 1 - X = 0.50067
largest repeated-value frequency = 3.00e-05
P(1/3 < X < 2/3) = 0.0000  (Cantor set omits this gap)
